In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os
import sys
import os

from dotenv import load_dotenv
# MODIFICATION : Importation pour interagir avec l'API Groq (au lieu d'OpenAI)
from groq import Groq

# --- Correction du chemin d'importation des modules 'src' ---
# Assure que le répertoire parent du projet est dans sys.path pour les imports de src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
# --- Fin de la correction ---
from src.rag_system import preprocess_query2
print("Modules importés avec succès.")

Modules importés avec succès.


In [ ]:
load_dotenv("../.env")

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
if os.getenv("GROQ_API_KEY") is None:
    print("Erreur : La variable d'environnement GROQ_API_KEY n'est pas définie.")
    print("Veuillez la définir avant d'exécuter ce notebook.")
    sys.exit(1)
else:
    print("Clé API GroqCloud configurée (via variable d'environnement ou directement).")

# Initialise le client Groq
# MODIFICATION : Création de l'objet client Groq
client = Groq()


Clé API GroqCloud configurée (via variable d'environnement ou directement).


In [17]:
faiss_index_path = "../data/embeddings/recipe_index.faiss"
metadata_path = "../data/embeddings/recipe_metadata.csv"
embedding_model_name = 'all-MiniLM-L6-v2' # Doit être le même que celui utilisé dans 03.
# MODIFICATION : Utilisation d'un modèle Llama 3 de GroqCloud
llm_model_name = 'meta-llama/llama-4-scout-17b-16e-instruct' # llama3-8b-8192 Tu peux aussi essayer 'llama3-70b-8192' pour un modèle plus puissant.

print(f"Chemin de l'index FAISS : {faiss_index_path}")
print(f"Chemin des métadonnées : {metadata_path}")
print(f"Modèle d'embedding utilisé : {embedding_model_name}")
print(f"Modèle LLM pour la génération : {llm_model_name}")



Chemin de l'index FAISS : ../data/embeddings/recipe_index.faiss
Chemin des métadonnées : ../data/embeddings/recipe_metadata.csv
Modèle d'embedding utilisé : all-MiniLM-L6-v2
Modèle LLM pour la génération : meta-llama/llama-4-scout-17b-16e-instruct


In [18]:
print(f"\nChargement du modèle d'embedding '{embedding_model_name}'...")
try:
    query_model = SentenceTransformer(embedding_model_name)
    print("Modèle d'embedding chargé avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement du modèle d'embedding : {e}")
    print("Assurez-vous d'avoir une connexion internet ou que le modèle est mis en cache.")
    sys.exit(1)



Chargement du modèle d'embedding 'all-MiniLM-L6-v2'...
Modèle d'embedding chargé avec succès.


In [19]:
print("\nChargement de l'index FAISS et des métadonnées...")
try:
    index = faiss.read_index(faiss_index_path)
    df_metadata = pd.read_csv(metadata_path)
    print(f"Index FAISS chargé. Nombre de vecteurs : {index.ntotal}")
    print(f"Métadonnées chargées. Nombre de recettes : {len(df_metadata)}")

    # Vérification que l'ordre des métadonnées correspond à l'index FAISS
    if index.ntotal != len(df_metadata):
        print("Avertissement : Le nombre de vecteurs dans l'index FAISS ne correspond pas au nombre de recettes dans les métadonnées.")
        print("Cela pourrait entraîner des résultats incorrects. Veuillez vérifier le processus de génération des embeddings.")

except FileNotFoundError as e:
    print(f"Erreur : Un fichier nécessaire n'a pas été trouvé : {e}")
    print("Assurez-vous d'avoir exécuté le notebook 03_embedding_generation.ipynb et que les chemins sont corrects.")
    sys.exit(1)
except Exception as e:
    print(f"Erreur lors du chargement de l'index ou des métadonnées : {e}")
    sys.exit(1)



Chargement de l'index FAISS et des métadonnées...


Index FAISS chargé. Nombre de vecteurs : 232111
Métadonnées chargées. Nombre de recettes : 232111


In [20]:
def search_recipes(query, top_k=5):
    """
    Recherche les recettes les plus similaires à une requête donnée.

    Args:
        query (str): La requête textuelle de l'utilisateur.
        top_k (int): Le nombre de recettes les plus similaires à retourner.

    Returns:
        pd.DataFrame: Un DataFrame contenant les recettes les plus similaires.
    """
    # NOUVEAU : Appliquer preprocess_query2 à la requête AVANT l'encodage
    processed_query = preprocess_query2(query)
    print(f"Requête originale: '{query}'")
    print(f"Requête prétraitée: '{processed_query}'") # Pour voir l'effet du prétraitement

    # 1. Encoder la requête prétraitée
    query_embedding = query_model.encode([processed_query], convert_to_numpy=True, normalize_embeddings=True)

    # 2. Rechercher dans l'index FAISS
    distances, indices = index.search(query_embedding, top_k)

    # 3. Récupérer les détails des recettes correspondantes
    retrieved_recipes = df_metadata.iloc[indices[0]].copy()

    # Ajouter la distance (ou score de similarité) aux résultats pour information
    retrieved_recipes['faiss_distance'] = distances[0]
    retrieved_recipes = retrieved_recipes.sort_values(by='faiss_distance').reset_index(drop=True)

    return retrieved_recipes

In [21]:
# %%
def generate_rag_answer(query, top_k=5):
    """
    Exécute un pipeline RAG :
    1. Récupère les recettes pertinentes à l'aide de FAISS.
    2. Construit un prompt pour le LLM en utilisant la requête et les recettes récupérées.
    3. Appelle le LLM pour générer une réponse.

    Args:
        query (str): La question de l'utilisateur.
        top_k (int): Le nombre de documents à récupérer.

    Returns:
        tuple: (Réponse du LLM, pd.DataFrame des documents sources)
    """
    print(f"\n--- Processus RAG pour la question : '{query}' ---")
    print("1. Récupération des recettes pertinentes...")
    # Récupération des documents (utilise la fonction search_recipes définie plus haut)
    retrieved_docs_df = search_recipes(query, top_k=top_k)

    if retrieved_docs_df.empty:
        print("Aucune recette pertinente trouvée.")
        return "Je n'ai pas d'information suffisante dans les recettes disponibles pour répondre à cette question.", pd.DataFrame()

    # 2. Construction du Contexte pour le LLM
    context_parts = []
    for idx, row in retrieved_docs_df.iterrows():
        # Utilise la colonne 'document' qui contient le texte prétraité de la recette
        context_parts.append(f"Recette {idx+1} (Nom: {row['name']}):\n{row['document']}")

    # Concaténer tous les documents pertinents en un seul contexte
    context = "\n\n".join(context_parts)

    print(f"2. Contexte construit à partir de {len(retrieved_docs_df)} recettes.")

    # 3. Construction du Prompt pour le LLM
    # Un prompt structuré pour guider le LLM.
    # Il est crucial d'instruire le LLM à n'utiliser que le contexte fourni.
    prompt_template = f"""
You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide the best possible answer(don't mention that you have used your general knowledge) — but always prioritize and integrate any useful information found in the recipes.

Do not mention recipe numbers, IDs, or refer to the recipes explicitly. Just answer the question as if you're drawing from your own expertise.

Recipe Context:
{context}

Question: {query}

Answer:
"""





    # NOUVEAU : Afficher le prompt complet avant l'appel au LLM
    print("\n" + "="*20 + " PROMPT COMPLET ENVOYÉ AU LLM " + "="*20)
    print(prompt_template)
    print("="*20 + " FIN DU PROMPT " + "="*20 + "\n")

    # 4. Appel au LLM de GroqCloud
    print("3. Appel au modèle de langage (LLM) de GroqCloud pour générer la réponse...")
    try:
        response = client.chat.completions.create(
            model=llm_model_name,
            messages=[
                {"role": "system", "content": "Tu es un assistant culinaire utile qui répond aux questions sur les recettes marocaines, en utilisant uniquement le contexte fourni."},
                {"role": "user", "content": prompt_template}
            ],
            temperature=0.9, # Température basse pour des réponses plus factuelles
            max_tokens=500 # Limiter la longueur de la réponse
        )
        llm_answer = response.choices[0].message.content
        print("4. Réponse du LLM reçue.")
        # NOUVEAU : Afficher la réponse complète du LLM
        print("\n" + "="*20 + " RÉPONSE COMPLÈTE DU LLM " + "="*20)
        print(llm_answer)
        print("="*20 + " FIN DE LA RÉPONSE " + "="*20 + "\n")

        return llm_answer, retrieved_docs_df
    except Exception as e:
        print(f"Erreur lors de l'appel au LLM de GroqCloud : {e}")
        return "Désolé, une erreur est survenue lors de la génération de la réponse. Veuillez réessayer.", pd.DataFrame()


In [22]:
question2 = "Comment préparer un couscous traditionnel marocain ?"
answer2, sources2 = generate_rag_answer(question2, top_k=3)

print(f"\n--- Question : {question2} ---")
print(f"Réponse du LLM :\n{answer2}")
print("\nRecettes sources utilisées (noms) :")
for idx, row in sources2.iterrows():
    print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
print("-" * 70)


--- Processus RAG pour la question : 'Comment préparer un couscous traditionnel marocain ?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'Comment préparer un couscous traditionnel marocain ?'
Requête prétraitée: 'comment traditionnel un couscous préparer ? marocain north african grain moroccan maghreb semolina'
2. Contexte construit à partir de 3 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you m

In [23]:
# %%
# Exemple 7 : Question sur une recette non-marocaine (attend une réponse d'absence d'information)
question7 = "How do I make Italian lasagna with meat sauce?"
answer7, sources7 = generate_rag_answer(question7, top_k=3) # Un top_k faible pour montrer qu'il n'y a rien de pertinent

print(f"\n--- Question : {question7} (Non-Marocaine) ---")
print(f"Réponse du LLM :\n{answer7}")
print("\nRecettes sources utilisées (noms) :")
if not sources7.empty:
    for idx, row in sources7.iterrows():
        print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
else:
    print("Aucune recette source pertinente trouvée ou utilisée.")
print("-" * 70)


--- Processus RAG pour la question : 'How do I make Italian lasagna with meat sauce?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'How do I make Italian lasagna with meat sauce?'
Requête prétraitée: 'with i lasagna meat make sauce? do how italian'
2. Contexte construit à partir de 3 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide the best possi

In [24]:
# Exemple 8: Demander des recettes végétariennes/végétaliennes marocaines (si présentes)
question8 = "Quelles sont les recettes marocaines végétariennes que je peux préparer ?"
answer8, sources8 = generate_rag_answer(question8, top_k=5)

print(f"\n--- Question : {question8} ---")
print(f"Réponse du LLM :\n{answer8}")
print("\nRecettes sources utilisées (noms) :")
if not sources8.empty:
    for idx, row in sources8.iterrows():
        print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
else:
    print("Aucune recette source pertinente trouvée.")
print("-" * 70)


--- Processus RAG pour la question : 'Quelles sont les recettes marocaines végétariennes que je peux préparer ?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'Quelles sont les recettes marocaines végétariennes que je peux préparer ?'
Requête prétraitée: 'les peux végétariennes recettes je quelles marocaines préparer ? que sont'
2. Contexte construit à partir de 5 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address th

In [25]:
# Exemple 13 : Question de culture générale (histoire)
question13 = "Qui a inventé l'ampoule électrique ?"
answer13, sources13 = generate_rag_answer(question13, top_k=2) # top_k faible car aucune pertinence attendue

print(f"\n--- Question : {question13} ---")
print(f"Réponse du LLM :\n{answer13}")
print("\nRecettes sources utilisées (noms) :")
if not sources13.empty:
    for idx, row in sources13.iterrows():
        print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
else:
    print("Aucune recette source pertinente trouvée.")
print("-" * 70)



--- Processus RAG pour la question : 'Qui a inventé l'ampoule électrique ?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'Qui a inventé l'ampoule électrique ?'
Requête prétraitée: 'a l'ampoule inventé électrique ? qui'
2. Contexte construit à partir de 2 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide the best possible answer(don't mention that 

In [26]:
# Exemple 17 : Question sur une autre cuisine (très distincte)
question17 = "Quels sont les ingrédients principaux d'un sushi ?"
answer17, sources17 = generate_rag_answer(question17, top_k=2)

print(answer17)
#print(f"\n--- Question : {question17} ---")
#print(f"Réponse du LLM :\n{answer17}")
#print("\nRecettes sources utilisées (noms) :")
#if not sources17.empty:
    #for idx, row in sources17.iterrows():
        #print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
#else:
    #print("Aucune recette source pertinente trouvée.")
#print("-" * 70)



--- Processus RAG pour la question : 'Quels sont les ingrédients principaux d'un sushi ?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'Quels sont les ingrédients principaux d'un sushi ?'
Requête prétraitée: 'les sushi principaux ? d'un quels ingrédients sont'
2. Contexte construit à partir de 2 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide th

In [27]:
# Exemple 17 : Question sur une autre cuisine (très distincte)
question17 = "how to make Chebakia marocaine(morocain cuisine)"
answer17, sources17 = generate_rag_answer(question17, top_k=5)

print(f"\n--- Question : {question17} ---")
print(f"Réponse du LLM :\n{answer17}")
print("\nRecettes sources utilisées (noms) :")
if not sources17.empty:
    for idx, row in sources17.iterrows():
        print(f"- {row.get('name', 'Nom inconnu')} (Distance FAISS: {row['faiss_distance']:.4f})")
else:
    print("Aucune recette source pertinente trouvée.")
print("-" * 70)



--- Processus RAG pour la question : 'how to make Chebakia marocaine(morocain cuisine)' ---
1. Récupération des recettes pertinentes...
Requête originale: 'how to make Chebakia marocaine(morocain cuisine)'
Requête prétraitée: 'marocaine(morocain make chebakia to cuisine) how'
2. Contexte construit à partir de 5 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide the best

In [30]:
question70 = "Comment préparer un sandwich de poulet ?"
answer70, sources70 = generate_rag_answer(question70, top_k=3)


--- Processus RAG pour la question : 'Comment préparer un sandwich de poulet ?' ---
1. Récupération des recettes pertinentes...
Requête originale: 'Comment préparer un sandwich de poulet ?'
Requête prétraitée: 'comment un préparer ? poulet sandwich de'
2. Contexte construit à partir de 3 recettes.

==================== PROMPT COMPLET ENVOYÉ AU LLM ====================

You are a helpful culinary assistant who answers cooking-related questions using recipe information as your primary reference.

Answer the question (the Question below that a user has writen to you . it's the only output tha he has given you this whole prompt and context it's just me )naturally and completely, without referencing that you're using a context or dataset. Combine and synthesize useful details from the recipes provided to give a clear, informative, and engaging response.

If the recipes do not fully address the question, you may use your general culinary knowledge to provide the best possible answer(don't m